# Extract traces + empirical errors for all test-data FLTs

This notebook:
- builds trace stacks for the three test-data FLT files,
- estimates empirical errors from the trace stacks,
- writes the results into the `SCI` extension as `ERROR_EMPIRICAL`,
- and produces diagnostic plots for each dataset.

In [1]:
%matplotlib widget

import os
from pathlib import Path
from importlib import reload

import numpy as np
from astropy.io import fits
from matplotlib import pyplot as plt

from airglow import extraction_utils as extract_utils

# Optional: set CRDS paths if they are not already configured.
# Update CRDS_PATH to match your local cache location if needed.
CRDS_PATH = "/Users/parke/crds_cache"
if "CRDS_PATH" not in os.environ:
    os.environ["CRDS_PATH"] = CRDS_PATH
    os.environ.setdefault("CRDS_SERVER_URL", "https://hst-crds.stsci.edu")
    os.environ.setdefault("iref", f"{CRDS_PATH}/references/hst/iref/")
    os.environ.setdefault("jref", f"{CRDS_PATH}/references/hst/jref/")
    os.environ.setdefault("oref", f"{CRDS_PATH}/references/hst/oref/")
    os.environ.setdefault("lref", f"{CRDS_PATH}/references/hst/lref/")
    os.environ.setdefault("nref", f"{CRDS_PATH}/references/hst/nref/")
    os.environ.setdefault("uref", f"{CRDS_PATH}/references/hst/uref/")

/Users/parke/mamba/envs/stenv/lib/python3.13/site-packages/stsci/tools/nmpfit.py:8: UserWarning: NMPFIT is deprecated - stsci.tools v 3.5 is the last version to contain it.
  warnings.warn("NMPFIT is deprecated - stsci.tools v 3.5 is the last version to contain it.")
/Users/parke/mamba/envs/stenv/lib/python3.13/site-packages/stsci/tools/gfit.py:18: UserWarning: GFIT is deprecated - stsci.tools v 3.4.12 is the last version to contain it.Use astropy.modeling instead.
  warnings.warn("GFIT is deprecated - stsci.tools v 3.4.12 is the last version to contain it."


The following tasks in the stistools package can be run with TEAL:
   basic2d      calstis     ocrreject     wavecal        x1d          x2d


In [3]:
TEST_DATA_DIR = extract_utils.find_test_data_dir()
flt_files = sorted(TEST_DATA_DIR.glob("*_flt.fits"))

X1D_PARAMS = extract_utils.default_x1d_params
STEP = float(X1D_PARAMS["extrsize"])  # pixels

print("FLT files:")
for f in flt_files:
    print(f"- {f.name}")

FLT files:
- of9b05010_flt.fits
- of9b05020_flt.fits
- of9b05030_flt.fits


In [5]:
results = {}
for fltfile in flt_files:
    output_x1d = fltfile.with_name(fltfile.name.replace("_flt", "_x1d_traces"))

    result = extract_utils.build_trace_stack_x1d(
        fltfile=fltfile,
        output_x1d=output_x1d,
        step=STEP,
        x1d_params=X1D_PARAMS,
        spectrum_column="flux",
    )

    results[fltfile.name] = {
        "fltfile": fltfile,
        "output_x1d": output_x1d,
        "trace_result": result,
    }



*** CALSTIS-6 -- Version 3.4.2 (19-Jan-2018) ***
Begin    10-Feb-2026 19:09:45 PST

INFO     'D1' has been appended to aperture
Warning  Grating-aperture throughput correction table (GACTAB) was not found,
         and no gac corrections will be applied
Input    /Users/parke/Repos/airglow/test-data/of9b05010_flt.fits
Output   /var/folders/6l/1fvgkb5d08b8tf7zpvnzkd7c0000gn/T/tmpzy1knojd/of9b05010_flt_y0000_x1d.fits
Rootname of9b05010
OBSMODE  TIME-TAG
APERTURE 52X0.2D1
OPT_ELEM G140M
DETECTOR FUV-MAMA

XTRACTAB oref$n7p10323o_1dx.fits
XTRACTAB PEDIGREE=INFLIGHT 29/05/97
XTRACTAB DESCRIP =Analysis from prop. 7064 and ground data
XTRACTAB DESCRIP =Analysis from prop. 7064
SPTRCTAB oref$77o1827do_1dt.fits
SPTRCTAB PEDIGREE=INFLIGHT 27/02/1997 01/12/2009
SPTRCTAB DESCRIP =New traces for select echelle secondary modes

Imset 1  Begin 19:09:45 PST
         Input read into memory.
Order 1  Begin 19:09:45 PST
X1DCORR  PERFORM
BACKCORR PERFORM
******** Calling Slfit ***********BACKCORR COMPLETE

In [9]:
reload(extract_utils)

for key, item in results.items():
    output_x1d = item["output_x1d"]

    revised = extract_utils.revise_background_error_in_x1d(
        x1dfile=output_x1d,
        output_x1d=output_x1d,
        align_centroids=True,
    )

    item["revised"] = revised

    print(f"Processed: {key} -> {output_x1d.name}")

Processed: of9b05010_flt.fits -> of9b05030_x1d_traces.fits


ValueError: name already used as a name or title

In [ ]:
# Plot extraction locations for each dataset.
for key, item in results.items():
    fltfile = item["fltfile"]
    y_positions = item["trace_result"]["y_positions"]

    data = fits.getdata(fltfile, 1)
    ny, nx = data.shape
    extrsize = float(X1D_PARAMS["extrsize"])

    fig, ax = plt.subplots()
    ax.set_title(f"Extraction locations: {fltfile.name}")
    ax.imshow(np.cbrt(data), aspect="auto")

    colors = plt.cm.viridis(np.linspace(0, 1, len(y_positions)))
    for y, color in zip(y_positions, colors):
        ax.axhspan(y - extrsize / 2, y + extrsize / 2, color=color, alpha=0.15)
        ax.plot([0, nx - 1], [y, y], color=color, lw=0.6)

    ax.set_xlim(0, nx - 1)
    ax.set_ylim(ny - 1, 0)

In [ ]:
# Plot 1D trace spectra for each dataset (color gradient by extraction location).
for key, item in results.items():
    output_x1d = item["output_x1d"]

    with fits.open(output_x1d) as hdul:
        trace_table = hdul["TRACES"].data
        traces = extract_utils._get_column(trace_table, "flux")
        wavelength = extract_utils._get_column(trace_table, "wavelength")

    if traces is None or wavelength is None:
        raise ValueError(f"Missing FLUX/WAVELENGTH columns in {output_x1d.name}.")

    colors = plt.cm.viridis(np.linspace(0, 1, traces.shape[0]))
    fig, ax = plt.subplots()
    ax.set_title(f"Trace spectra: {output_x1d.name}")
    for spectrum, wave, color in zip(traces, wavelength, colors):
        ax.plot(wave, spectrum, color=color, alpha=0.8, lw=0.7)
    ax.set_xlabel("Wavelength")
    ax.set_ylabel("Flux")

In [ ]:
# Plot FLUX, ERROR, and ERROR_EMPIRICAL from SCI for each dataset.
for key, item in results.items():
    output_x1d = item["output_x1d"]

    with fits.open(output_x1d) as hdul:
        sci = hdul["SCI"] if "SCI" in hdul else hdul[1]
        flux = extract_utils._get_column(sci.data, "flux")
        error = extract_utils._get_column(sci.data, "error")
        error_empirical = extract_utils._get_column(sci.data, "error_empirical")
        wavelength = extract_utils._get_column(sci.data, "wavelength")

    if flux is None or error is None or wavelength is None:
        raise ValueError(f"Missing FLUX/ERROR/WAVELENGTH in {output_x1d.name}.")
    if error_empirical is None:
        raise ValueError(f"Missing ERROR_EMPIRICAL in {output_x1d.name}.")

    if np.ndim(wavelength) > 1:
        wavelength = wavelength[0]
    if np.ndim(flux) > 1:
        flux = flux[0]
    if np.ndim(error) > 1:
        error = error[0]
    if np.ndim(error_empirical) > 1:
        error_empirical = error_empirical[0]

    fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
    fig.suptitle(f"SCI: {output_x1d.name}")

    axes[0].plot(wavelength, flux, color="tab:blue", lw=0.8)
    axes[0].set_ylabel("Flux")

    axes[1].plot(wavelength, error, color="tab:orange", lw=0.8)
    axes[1].set_ylabel("Error")

    axes[2].plot(wavelength, error_empirical, color="tab:green", lw=0.8)
    axes[2].set_ylabel("Error Empirical")
    axes[2].set_xlabel("Wavelength")

    plt.tight_layout()